<a href="https://colab.research.google.com/github/Datahuntl/Estudo-Comparativo-de-Detec-o-de-DeepFakes/blob/main/XceptionNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install timm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
import os
import numpy as np
from PIL import Image
from google.colab import drive
import timm

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando na instância: {device}")

Rodando na instância: cpu


In [ ]:
import shutil
import os

print("📦 Copiando o arquivo ZIP do Drive para o disco local do Colab...")
shutil.copy("/content/drive/MyDrive/IC DeepFakes/DATASET_FACES.zip", "/content/DATASET_FACES.zip")

print("🔓 Extraindo os arquivos no SSD local...")
!unzip -q /content/DATASET_FACES.zip -d /content/dataset_local/

print("✅ Concluído!")

📦 Copiando o arquivo ZIP do Drive para o disco local do Colab...
🔓 Extraindo os arquivos no SSD local...
✅ Concluído!


In [ ]:
PATH_REAL = "/content/dataset_local/DATASET_FACES/REAL"
PATH_FAKE = "/content/dataset_local/DATASET_FACES/FAKE"

In [ ]:
class DeepfakeFaceDataset(Dataset):
    def __init__(self, real_face_dir, fake_face_dir, transform=None):
        self.real_images = [os.path.join(real_face_dir, img) for img in os.listdir(real_face_dir) if img.lower().endswith(('jpg', 'jpeg', 'png'))]
        self.fake_images = [os.path.join(fake_face_dir, img) for img in os.listdir(fake_face_dir) if img.lower().endswith(('jpg', 'jpeg', 'png'))]

        self.all_images = self.real_images + self.fake_images
        self.labels = [0] * len(self.real_images) + [1] * len(self.fake_images)
        self.transform = transform

    def __len__(self):
        return len(self.all_images)

    def __getitem__(self, idx):
        img_path = self.all_images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
        return image, label

# --- CONFIGURAÇÃO ESPECÍFICA DA XCEPTION ---
transform_xception = transforms.Compose([
    transforms.Resize((299, 299)),  # Resolução exigida pela arquitetura Xception
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

full_dataset = DeepfakeFaceDataset(real_face_dir=PATH_REAL, fake_face_dir=PATH_FAKE, transform=transform_xception)

if len(full_dataset) == 0:
    print("\n❌ ERRO CRÍTICO: Nenhuma foto foi localizada nas pastas do Drive. Verifique os caminhos.")
else:
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

    print(f"\n✅ Dataset Xception carregado com sucesso!")
    print(f"Total de imagens encontradas: {len(full_dataset)} | Treino: {train_size} | Validação: {val_size}")


✅ Dataset Xception carregado com sucesso!
Total de imagens encontradas: 48979 | Treino: 39183 | Validação: 9796


In [ ]:
# Cria a rede Xception carregando os pesos oficiais do ImageNet
model = timm.create_model('xception', pretrained=True)

# Substitui a camada totalmente conectada (fc) original por uma saída binária com ativação Sigmoide
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 1),
    nn.Sigmoid()
)

model = model.to(device)
print("✅ Modelo XceptionNet inicializado com pesos pré-treinados e pronto para Transfer Learning!")

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
✅ Modelo XceptionNet inicializado com pesos pré-treinados e pronto para Transfer Learning!


In [ ]:
import matplotlib.pyplot as plt
import time
from tqdm.auto import tqdm

# --- CONFIGURAÇÃO DE DIRETÓRIOS EXCLUSIVOS DA XCEPTION ---
CHECKPOINT_DIR = "/content/drive/MyDrive/IC DeepFakes/XceptionNet_Outputs"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

checkpoint_path = os.path.join(CHECKPOINT_DIR, "xception_best_checkpoint.pth")
grafico_loss_path = os.path.join(CHECKPOINT_DIR, "curva_perda_xception.png")
grafico_acc_path = os.path.join(CHECKPOINT_DIR, "curva_acuracia_xception.png")

# --- HIPERPARÂMETROS ---
criterion = nn.BCELoss()
# Learning rate menor (0.0001) para preservar os pesos congelados da rede pré-treinada
optimizer = optim.Adam(model.parameters(), lr=0.0001)
epochs = 2
start_epoch = 0
best_val_acc = 0.0

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

# --- SEÇÃO DE RECUPERAÇÃO AUTOMÁTICA (RESUME) ---
if os.path.exists(checkpoint_path):
    print(f"🔄 Checkpoint da Xception encontrado em: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint['best_val_acc']
    history = checkpoint['history']
    print(f"▶️ Resumindo treinamento a partir da Época [{start_epoch+1}/{epochs}] | Melhor Acc: {best_val_acc:.4f}\n")
else:
    print("🆕 Nenhum checkpoint encontrado para Xception. Iniciando do zero.\n")

# --- LOOP COMPLETO ---
print("🚀 Iniciando o treinamento da XceptionNet...\n")

for epoch in range(start_epoch, epochs):
    start_time = time.time()

    # Treino
    model.train()
    train_loss = 0.0
    all_train_labels, all_train_preds = [], []
    train_bar = tqdm(train_loader, desc=f"🎬 Época {epoch+1}/{epochs} [Treino Xception]", leave=False)

    for batch_idx, (inputs, labels) in enumerate(train_bar):
        inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        train_loss += current_loss * inputs.size(0)

        preds_binary = (outputs.cpu().detach().numpy() > 0.5).astype(int)
        all_train_labels.extend(labels.cpu().numpy())
        all_train_preds.extend(preds_binary)

        if batch_idx % 2 == 0:
            batch_acc = accuracy_score(labels.cpu().numpy(), preds_binary)
            train_bar.set_postfix({"Loss": f"{current_loss:.4f}", "Acc": f"{batch_acc*100:.1f}%"})

    epoch_train_loss = train_loss / len(train_loader.dataset)
    epoch_train_acc = accuracy_score(all_train_labels, all_train_preds)

    # Validação
    model.eval()
    val_loss = 0.0
    all_labels, all_preds = [], []
    val_bar = tqdm(val_loader, desc=f"🔍 Época {epoch+1}/{epochs} [Validação Xception]", leave=False)

    with torch.no_grad():
        for inputs, labels in val_bar:
            inputs, labels_eval = inputs.to(device), labels.to(device).float().unsqueeze(1)
            outputs = model(inputs)

            loss = criterion(outputs, labels_eval)
            val_loss += loss.item() * inputs.size(0)

            all_labels.extend(labels.numpy())
            all_preds.extend(outputs.cpu().numpy())

    epoch_val_loss = val_loss / len(val_loader.dataset)
    all_labels, all_preds = np.array(all_labels), np.array(all_preds)
    binary_preds = (all_preds > 0.5).astype(int)

    # Métricas Científicas
    epoch_val_acc = accuracy_score(all_labels, binary_preds)
    f1 = f1_score(all_labels, binary_preds, zero_division=0)
    prec = precision_score(all_labels, binary_preds, zero_division=0)
    rec = recall_score(all_labels, binary_preds, zero_division=0)
    try: auc = roc_auc_score(all_labels, all_preds)
    except: auc = 0.5

    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_acc"].append(epoch_val_acc)

    epoch_time = time.time() - start_time
    print(f"📊 [FIM DA ÉPOCA {epoch+1}/{epochs}] Tempo: {epoch_time:.1f}s")
    print(f"   📈 Treino     -> Perda: {epoch_train_loss:.4f} | Acurácia: {epoch_train_acc*100:.2f}%")
    print(f"   📉 Validação  -> Perda: {epoch_val_loss:.4f} | Acurácia: {epoch_val_acc*100:.2f}%")
    print(f"   ✨ Métricas   -> AUC-ROC: {auc:.4f} | F1-Score: {f1:.4f} | Precisão: {prec:.4f} | Revocação: {rec:.4f}")

    # Salvamento de Checkpoint Baseado em Desempenho
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        print(f"   💾 ⭐ Nova melhor acurácia obtida! Salvando pesos no Drive...")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
            'history': history
        }, checkpoint_path)
    print("-" * 80)

print("🏆 Treinamento da XceptionNet Concluído!")

🆕 Nenhum checkpoint encontrado para Xception. Iniciando do zero.

🚀 Iniciando o treinamento da XceptionNet...



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


🎬 Época 1/2 [Treino Xception]:   0%|          | 0/2449 [00:00<?, ?it/s]

In [ ]:
print("\nExportando gráficos de desempenho...")

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(history["train_loss"])+1), history["train_loss"], label="Treino (Loss)", color="blue", linestyle="--")
plt.plot(range(1, len(history["val_loss"])+1), history["val_loss"], label="Validação (Loss)", color="red")
plt.title("Evolução da Perda - XceptionNet")
plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.savefig(grafico_loss_path, dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(history["train_acc"])+1), history["train_acc"], label="Treino (Acc)", color="blue", linestyle="--")
plt.plot(range(1, len(history["val_acc"])+1), history["val_acc"], label="Validação (Acc)", color="green")
plt.title("Evolução da Acurácia - XceptionNet")
plt.xlabel("Épocas")
plt.ylabel("Acurácia")
plt.legend()
plt.grid(True)
plt.savefig(grafico_acc_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✨ Artefatos visuais salvos na pasta: {CHECKPOINT_DIR}")